**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/03-calibrate-and-undistort.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/calibrate-camera) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6C · Calibrate & Undistort

_Lecture 6, Section 5: calibrateCamera, reprojection error and undistortion._

Calibrate the Udacity GoPro, judge the result with the **reprojection error**, remove bad views, undistort
images, choose between cropping and keeping borders, and save the calibration to a YAML file.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('gopro')
print("data ready")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=3, suppress=True)

nx, ny = 8, 6
objp = C.board_points(nx, ny)                      # what you built in 6B
images = sorted(glob.glob("data/calibration_wide/GOPR*.jpg"))
objpoints, imgpoints, used = [], [], []
for fname in images:
    gray = cv2.imread(fname, cv2.IMREAD_GRAYSCALE)
    ok, corners = C.find_corners(gray, (nx, ny))   # findChessboardCorners + cornerSubPix, shape (N, 1, 2)
    if ok:
        objpoints.append(objp); imgpoints.append(corners); used.append(fname)
print(len(used), "views, image size", gray.shape)

## 1. `cv2.calibrateCamera`

```python
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, image_size, None, None)
```

`image_size` is **(width, height)**. `gray.shape` is (height, width), hence `gray.shape[::-1]`.

In [ ]:
image_size = None   # TODO
ret, mtx, dist, rvecs, tvecs = None, None, None, None, None   # TODO: calibrate

print("RMS reprojection error:", ret)
print("K =\n", mtx)
print("dist =", dist)

In [ ]:
C.check_calibration(ret, mtx, dist, image_size)

## 2. Quiz: `cal_undistort` (the Udacity quiz)

Write a function that takes an image, object points and image points, calibrates, undistorts and returns the
undistorted image.

In [ ]:
def cal_undistort(img, objpoints, imgpoints):
    # Use cv2.calibrateCamera() and cv2.undistort()
    undist = np.copy(img)  # Delete this line
    return undist

img = cv2.imread("data/calibration_wide/test_image.jpg")
undistorted = cal_undistort(img, objpoints, imgpoints)
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7.5))
ax1.imshow(img[:, :, ::-1]); ax1.set_title("Original Image", fontsize=24)
ax2.imshow(undistorted[:, :, ::-1]); ax2.set_title("Undistorted Image", fontsize=24)
plt.show()

In [ ]:
C.check_cal_undistort(cal_undistort)

> In a real system you calibrate **once**, save `mtx` and `dist`, and only call `cv2.undistort` per frame.

## 3. Per-view reprojection error

Project each view's object points with its `rvec, tvec` and compare to the detected corners:
`err_j = sqrt(mean(dx² + dy²))`. Fill in the loop.

In [ ]:
errs = []
for objp_j, imgp_j, r, t in zip(objpoints, imgpoints, rvecs, tvecs):
    # TODO: projected, _ = cv2.projectPoints(...); compute the RMS distance to imgp_j
    errs.append(0.0)

plt.figure(figsize=(14, 4)); plt.bar(range(len(errs)), errs, color="#28568a")
plt.axhline(ret, color="green", ls="--", label=f"overall RMS {ret:.2f}"); plt.xlabel("view"); plt.ylabel("px"); plt.legend(); plt.show()

In [ ]:
C.check_per_view(errs, len(objpoints))

## 4. Drop the bad views and recalibrate

Look at the worst view(s). Remove every view with an error above **1.5 px** from `objpoints` *and* `imgpoints`
and calibrate again.

In [ ]:
worst = int(np.argmax(errs))
plt.figure(figsize=(9, 7)); plt.imshow(cv2.imread(used[worst])[:, :, ::-1]); plt.title(f"worst view: {used[worst][-12:]} ({errs[worst]:.2f} px)"); plt.axis("off"); plt.show()

# TODO: keep only good views and recalibrate
ret2, mtx2, dist2 = ret, mtx, dist

print(f"RMS {ret:.3f} -> {ret2:.3f} px")

In [ ]:
C.check_recalibrated(ret, ret2)

## 5. How certain are the parameters? `calibrateCameraExtended`

In [ ]:
out = cv2.calibrateCameraExtended(objpoints, imgpoints, image_size, None, None)
rms_e, K_e, D_e, _, _, std_in, std_ex, per_view = out
names = ["fx", "fy", "cx", "cy", "k1", "k2", "p1", "p2", "k3"]
vals = [K_e[0, 0], K_e[1, 1], K_e[0, 2], K_e[1, 2], *D_e.ravel()[:5]]
for n, v, s in zip(names, vals, std_in.ravel()[:9]):
    print(f"{n:>3} = {v:9.4f} ± {s:.4f}")

## 6. Keep or crop the borders: `getOptimalNewCameraMatrix`

`alpha = 0` crops to valid pixels only; `alpha = 1` keeps all source pixels. On this wide lens `alpha = 1`
shows that the 5-coefficient polynomial **folds** outside the region covered by corners. The rational model
(`cv2.CALIB_RATIONAL_MODEL`, 8 coefficients) usually extrapolates better.

In [ ]:
h, w = img.shape[:2]
fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
for a, alpha in zip(ax[:2], (0.0, 1.0)):
    newK, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), alpha, (w, h))
    und = cv2.undistort(img, mtx, dist, None, newK)
    x, y, rw, rh = roi; cv2.rectangle(und, (x, y), (x + rw, y + rh), (0, 255, 0), 4)
    a.imshow(und[:, :, ::-1]); a.set_title(f"5 coefficients, alpha = {alpha}"); a.axis("off")

r8, K8, D8, _, _ = cv2.calibrateCamera(objpoints, imgpoints, (w, h), None, None, flags=cv2.CALIB_RATIONAL_MODEL)
newK8, _ = cv2.getOptimalNewCameraMatrix(K8, D8, (w, h), 1.0, (w, h))
ax[2].imshow(cv2.undistort(img, K8, D8, None, newK8)[:, :, ::-1]); ax[2].set_title(f"rational model (RMS {r8:.2f}), alpha = 1"); ax[2].axis("off")
plt.show()

## 7. Fast undistortion for video: compute the map once

In [ ]:
import time
map1, map2 = cv2.initUndistortRectifyMap(mtx, dist, None, mtx, (w, h), cv2.CV_16SC2)
t0 = time.perf_counter()
for _ in range(20): a = cv2.undistort(img, mtx, dist, None, mtx)
t1 = time.perf_counter()
for _ in range(20): b = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)
t2 = time.perf_counter()
print(f"undistort: {(t1 - t0) / 20 * 1000:.1f} ms/frame   remap: {(t2 - t1) / 20 * 1000:.1f} ms/frame")
print("max pixel difference:", int(np.abs(a.astype(int) - b).max()))

## 8. Save the calibration

In [ ]:
fs = cv2.FileStorage("gopro_camera.yaml", cv2.FILE_STORAGE_WRITE)
fs.write("image_width", w); fs.write("image_height", h)
fs.write("camera_matrix", mtx); fs.write("distortion_coefficients", dist); fs.write("rms", ret)
fs.release()
print(open("gopro_camera.yaml").read())

fs = cv2.FileStorage("gopro_camera.yaml", cv2.FILE_STORAGE_READ)
K_loaded = fs.getNode("camera_matrix").mat(); fs.release()
print("loaded K matches:", np.allclose(K_loaded, mtx))

## Your turn

1. Calibrate with only the **first 5** views, then 10, 20 and all. Plot fx and k1 against the number of views. When do
   they stop changing?
2. Calibrate with only **frontal** views (pick them by looking at `rvecs`: small rotation angle). What goes wrong, and why?
3. Undistort `test_image.jpg` with the rational model and `alpha = 0`. Is it better than the 5-coefficient result?
   Measure it: detect the corners in both undistorted images and check how straight the rows are
   (fit a line to each row and report the mean residual).